# CNV Preprocessing — TCGA-BRCA
Produces 8 experimental dataset variants in `statistical_filtered/`.

Notebook location: `Thesis_v3/01_Causal_feature_extraction/CNV/`

CNV-specific notes vs RNA pipeline:
- Missing values require explicit imputation (structural zeros in CNV assays)
- Z-score normalization applied **after** statistical filtering (not before)
- No log2 transform (CNV values are already log-ratio scale)

| V | Name | Selection logic |
|---|---|---|
| 1 | ultra_conservative | Variance > 98.3 percentile |
| 2 | conservative | Variance > 97.5 percentile |
| 3 | standard | Variance > 95.9 percentile |
| 4 | fdr_significant | Spearman FDR < 0.05, top 1000 by composite |
| 5 | balanced | Var top 25% then Spearman top 10% |
| 6 | correlation | Abs. Spearman > 97.5 percentile |
| 7 | top_correlated | Top 100 positive + top 100 negative Spearman |
| 8 | composite | Average rank of Spearman + MI + Distance corr |


In [2]:
from pathlib import Path

# Notebook: Thesis_v3/01_Causal_feature_extraction/CNV/cnv_preprocessing.ipynb
_cwd = Path().resolve()
BASE_DIR = next(
    p for p in [_cwd, *_cwd.parents]
    if (p / "data" / "TCGA-BRCA.gene-level_ascat2.tsv").exists()
)

CNV_FILE     = BASE_DIR / "data" / "TCGA-BRCA.gene-level_ascat2.tsv"
OUTCOME_FILE = BASE_DIR / "data" / "outcome.csv"
OUTPUT_DIR   = BASE_DIR / "01_Causal_feature_extraction" / "CNV" / "statistical_filtered"
STATS_CACHE  = BASE_DIR / "01_Causal_feature_extraction" / "CNV" / "cnv_statistics_cache.csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MIN_GENES    = 50
TARGET_BROAD = 1000

assert CNV_FILE.exists(),     f"Not found: {CNV_FILE}"
assert OUTCOME_FILE.exists(), f"Not found: {OUTCOME_FILE}"

print(f"Base   : {BASE_DIR}")
print(f"Output : {OUTPUT_DIR}")
print(f"Cache  : {STATS_CACHE}")
print("Paths OK")


Base   : C:\Users\olegk\Desktop\Thesis_v3
Output : C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction\CNV\statistical_filtered
Cache  : C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction\CNV\cnv_statistics_cache.csv
Paths OK


In [3]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr, rankdata
from statsmodels.stats.multitest import multipletests
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_regression
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings("ignore")


In [4]:
print("Loading CNV data...")
cnv_raw = pd.read_csv(CNV_FILE, sep="\t", index_col=0)
print(f"  Raw shape  : {cnv_raw.shape}  (genes x samples)")

cnv = cnv_raw.T.copy()

print("Loading outcome...")
outcome = pd.read_csv(OUTCOME_FILE, index_col=0)
print(f"  Samples    : {len(outcome)}")
print(f"  OS.time    : {outcome['OS.time'].min():.0f} - {outcome['OS.time'].max():.0f} days")
print(f"  Events     : {int(outcome['OS'].sum())} ({outcome['OS'].mean()*100:.1f}%)")

common  = sorted(set(cnv.index) & set(outcome.index))
cnv     = cnv.loc[common].copy()
outcome = outcome.loc[common].copy()
print(f"  Overlap    : {len(common)} samples")


Loading CNV data...
  Raw shape  : (60623, 1082)  (genes x samples)
Loading outcome...
  Samples    : 1076
  OS.time    : 1 - 8605 days
  Events     : 150 (13.9%)
  Overlap    : 1043 samples


In [5]:
print("Quality control and imputation...")

# Remove genes with >20% missing (CNV assay structural missingness)
miss_frac = cnv.isna().mean(axis=0)
cnv = cnv.loc[:, miss_frac <= 0.20]
print(f"  After missing filter (>20%)  : {cnv.shape[1]:,} genes")

# Median imputation for residual missing values
# Mean imputation would shift the CNV distribution; median is more conservative
imp = SimpleImputer(strategy="median")
cnv = pd.DataFrame(imp.fit_transform(cnv), index=cnv.index, columns=cnv.columns)
print(f"  After median imputation      : {cnv.isnull().sum().sum()} missing values remain")

# Remove zero-variance genes
var = cnv.var(axis=0)
cnv = cnv.loc[:, var > 0]
print(f"  After zero-variance removal  : {cnv.shape[1]:,} genes")


Quality control and imputation...
  After missing filter (>20%)  : 60,226 genes
  After median imputation      : 0 missing values remain
  After zero-variance removal  : 60,226 genes


In [6]:
print("Variance pre-filter: keep top 75%...")

var     = cnv.var(axis=0)
cnv_var = cnv.loc[:, var > var.quantile(0.25)]
print(f"  Before : {cnv.shape[1]:,}  After : {cnv_var.shape[1]:,} genes")
print()
print("Note: z-score normalization is applied per-dataset after statistical")
print("selection, not here. CNV values are already on a log-ratio scale.")


Variance pre-filter: keep top 75%...
  Before : 60,226  After : 45,169 genes

Note: z-score normalization is applied per-dataset after statistical
selection, not here. CNV values are already on a log-ratio scale.


In [7]:
if STATS_CACHE.exists():
    print(f"Loading cached statistics ({STATS_CACHE.name})...")
    stats = pd.read_csv(STATS_CACHE)
    print(f"  Loaded: {len(stats):,} genes")
else:
    import time
    os_time = outcome["OS.time"].values
    genes   = cnv_var.columns.tolist()
    n       = len(genes)
    print(f"Computing statistics for {n:,} genes...")

    # 1. Spearman correlation
    print("  [1/3] Spearman correlation...")
    t0 = time.time()
    rho_arr, pval_arr = [], []
    for i, g in enumerate(genes):
        r, p = spearmanr(cnv_var[g].values, os_time)
        rho_arr.append(0.0 if np.isnan(r) else float(r))
        pval_arr.append(1.0 if np.isnan(p) else float(p))
        if (i + 1) % 10000 == 0:
            print(f"    {i+1:,} / {n:,}  ({time.time()-t0:.0f}s)")
    _, fdr_arr, _, _ = multipletests(pval_arr, method="fdr_bh")

    # 2. Mutual information
    print("  [2/3] Mutual information...")
    mi_arr = mutual_info_regression(
        cnv_var.values, os_time, random_state=42, n_neighbors=5
    )

    # 3. Distance correlation (rank-based approximation)
    # Full O(n^2) distance correlation per gene is computationally prohibitive
    # at this scale. Pearson correlation on ranked values serves as a fast
    # monotone proxy; it diversifies the composite ranking from Spearman alone.
    print("  [3/3] Distance correlation (rank approximation)...")
    os_rank = rankdata(os_time).astype(float)
    dc_arr  = []
    for g in genes:
        x_rank = rankdata(cnv_var[g].values).astype(float)
        dc_arr.append(abs(float(np.corrcoef(x_rank, os_rank)[0, 1])))

    stats = pd.DataFrame({
        "gene":          genes,
        "spearman_rho":  rho_arr,
        "abs_spearman":  np.abs(rho_arr),
        "pval":          pval_arr,
        "fdr":           fdr_arr,
        "mutual_info":   mi_arr,
        "distance_corr": dc_arr,
        "variance":      cnv_var.var().values,
    })

    stats["rank_spearman"] = rankdata(-stats["abs_spearman"])
    stats["rank_mi"]       = rankdata(-stats["mutual_info"])
    stats["rank_dcor"]     = rankdata(-stats["distance_corr"])
    stats["composite"]     = (
        stats["rank_spearman"] + stats["rank_mi"] + stats["rank_dcor"]
    ) / 3.0
    stats = stats.sort_values("composite").reset_index(drop=True)

    stats.to_csv(STATS_CACHE, index=False)
    print(f"  Cached to {STATS_CACHE.name}")

n_fdr = int((stats["fdr"] < 0.05).sum())
print(f"  FDR < 0.05 : {n_fdr:,} / {len(stats):,} genes")
print(f"  Top 10 by composite score:")
print(stats[["gene","abs_spearman","mutual_info","distance_corr","composite"]].head(10).to_string(index=False))


Computing statistics for 45,169 genes...
  [1/3] Spearman correlation...
    10,000 / 45,169  (13s)
    20,000 / 45,169  (26s)
    30,000 / 45,169  (39s)
    40,000 / 45,169  (54s)
  [2/3] Mutual information...
  [3/3] Distance correlation (rank approximation)...
  Cached to cnv_statistics_cache.csv
  FDR < 0.05 : 0 / 45,169 genes
  Top 10 by composite score:
              gene  abs_spearman  mutual_info  distance_corr   composite
 ENSG00000253198.1      0.051041     0.039528       0.051041 1067.000000
 ENSG00000253845.1      0.051041     0.037812       0.051041 1095.666667
 ENSG00000221043.2      0.051535     0.034886       0.051535 1120.000000
 ENSG00000287323.1      0.051535     0.034860       0.051535 1121.333333
 ENSG00000273173.5      0.058211     0.026827       0.058211 1129.666667
 ENSG00000242928.3      0.051491     0.034424       0.051491 1142.000000
 ENSG00000199047.3      0.051491     0.032947       0.051491 1202.333333
ENSG00000153443.13      0.051786     0.031384       0.

In [8]:
print("Creating 8 dataset variants...")
print(f"Output: {OUTPUT_DIR}")
print("Z-score normalization applied to each variant after gene selection.")

scaler = StandardScaler()

def build(gene_list, min_g=MIN_GENES):
    """Select genes, pad if below minimum, apply z-score normalization."""
    gene_list = [g for g in gene_list if g in cnv_var.columns]
    if len(gene_list) < min_g:
        have = set(gene_list)
        gene_list += [g for g in stats["gene"] if g not in have][:min_g - len(gene_list)]
    subset = cnv_var[gene_list]
    # Z-score normalization applied after selection (CNV-specific design choice)
    normed = pd.DataFrame(
        scaler.fit_transform(subset),
        index=subset.index,
        columns=subset.columns
    )
    return normed

created = []

# V1 Ultra-Conservative: variance > 98.3 percentile
df    = build(stats[stats["variance"] > stats["variance"].quantile(0.983)]["gene"].tolist())
fname = f"cnv_1_ultra_conservative_{len(df.columns)}genes.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V1","name":"ultra_conservative","n":len(df.columns),"logic":"Variance > 98.3 pct","file":fname})
print(f"  V1 ultra_conservative : {len(df.columns):,} genes")

# V2 Conservative: variance > 97.5 percentile
df    = build(stats[stats["variance"] > stats["variance"].quantile(0.975)]["gene"].tolist())
fname = f"cnv_2_conservative_{len(df.columns)}genes.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V2","name":"conservative","n":len(df.columns),"logic":"Variance > 97.5 pct","file":fname})
print(f"  V2 conservative       : {len(df.columns):,} genes")

# V3 Standard: variance > 95.9 percentile
df    = build(stats[stats["variance"] > stats["variance"].quantile(0.959)]["gene"].tolist())
fname = f"cnv_3_standard_{len(df.columns)}genes.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V3","name":"standard","n":len(df.columns),"logic":"Variance > 95.9 pct","file":fname})
print(f"  V3 standard           : {len(df.columns):,} genes")

# V4 FDR-significant: Spearman FDR < 0.05, top TARGET_BROAD by composite
fdr_genes = stats[stats["fdr"] < 0.05].head(TARGET_BROAD)["gene"].tolist()
df    = build(fdr_genes, min_g=TARGET_BROAD)
fname = f"cnv_4_fdr_significant_{len(df.columns)}genes.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V4","name":"fdr_significant","n":len(df.columns),"logic":"FDR<0.05, top 1000 by composite","file":fname})
print(f"  V4 fdr_significant    : {len(df.columns):,} genes  (raw FDR<0.05: {int((stats['fdr']<0.05).sum())})")

# V5 Balanced: top 25% variance then top 10% Spearman within that subset
var_sub     = stats[stats["variance"] > stats["variance"].quantile(0.75)]
corr_thresh = var_sub["abs_spearman"].quantile(0.90)
df    = build(var_sub[var_sub["abs_spearman"] > corr_thresh]["gene"].tolist())
fname = f"cnv_5_balanced_{len(df.columns)}genes.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V5","name":"balanced","n":len(df.columns),"logic":"Var top 25% -> Spearman top 10%","file":fname})
print(f"  V5 balanced           : {len(df.columns):,} genes")

# V6 Correlation-based: abs Spearman > 97.5 percentile
df    = build(stats[stats["abs_spearman"] > stats["abs_spearman"].quantile(0.975)]["gene"].tolist())
fname = f"cnv_6_correlation_{len(df.columns)}genes.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V6","name":"correlation","n":len(df.columns),"logic":"Abs Spearman > 97.5 pct","file":fname})
print(f"  V6 correlation        : {len(df.columns):,} genes")

# V7 Top Correlated: top 100 positive + top 100 negative Spearman
# Using 100+100 to match RNA pipeline (old CNV used 20+20 which was too compact)
top_pos = stats[stats["spearman_rho"] > 0].head(100)["gene"].tolist()
top_neg = stats[stats["spearman_rho"] < 0].tail(100)["gene"].tolist()
df      = build(sorted(set(top_pos + top_neg)))
stats[stats["gene"].isin(set(top_pos + top_neg))][
    ["gene","spearman_rho","pval","fdr","variance"]
].to_csv(OUTPUT_DIR / "cnv_7_top_correlated_annotated.csv", index=False)
fname = f"cnv_7_top_correlated_{len(df.columns)}genes.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V7","name":"top_correlated","n":len(df.columns),"logic":"Top 100 pos + top 100 neg Spearman","file":fname})
print(f"  V7 top_correlated     : {len(df.columns):,} genes")

# V8 Composite: top TARGET_BROAD by average rank of Spearman + MI + Dcor
df    = build(stats.head(TARGET_BROAD)["gene"].tolist(), min_g=TARGET_BROAD)
fname = f"cnv_8_composite_{len(df.columns)}genes.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V8","name":"composite","n":len(df.columns),"logic":"Avg rank Spearman+MI+Dcor, top 1000","file":fname})
print(f"  V8 composite          : {len(df.columns):,} genes")

# Auxiliary files
outcome.to_csv(OUTPUT_DIR / "outcome.csv")
stats.to_csv(OUTPUT_DIR / "cnv_statistics_complete.csv", index=False)
summary = pd.DataFrame(created)
summary.to_csv(OUTPUT_DIR / "datasets_summary.csv", index=False)

print()
print(summary[["V","name","n","logic"]].to_string(index=False))
print(f"Saved to: {OUTPUT_DIR}")


Creating 8 dataset variants...
Output: C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction\CNV\statistical_filtered
Z-score normalization applied to each variant after gene selection.
  V1 ultra_conservative : 768 genes
  V2 conservative       : 1,128 genes
  V3 standard           : 1,813 genes
  V4 fdr_significant    : 1,000 genes  (raw FDR<0.05: 0)
  V5 balanced           : 1,130 genes
  V6 correlation        : 1,127 genes
  V7 top_correlated     : 200 genes
  V8 composite          : 1,000 genes

 V               name    n                               logic
V1 ultra_conservative  768                 Variance > 98.3 pct
V2       conservative 1128                 Variance > 97.5 pct
V3           standard 1813                 Variance > 95.9 pct
V4    fdr_significant 1000     FDR<0.05, top 1000 by composite
V5           balanced 1130     Var top 25% -> Spearman top 10%
V6        correlation 1127             Abs Spearman > 97.5 pct
V7     top_correlated  200  Top 100 pos + top